# Apply A BB1 Pi Waveform To The Pulse PhMod Template


## 1. Parameters

Edit the actual BB1 pi pulse duration and the phase-modulation dwell time. The notebook saves `t_pulse = TARGET_PULSE_DURATION_US - t_mod_offset`.


In [ ]:
TARGET_PULSE_DURATION_US = 20
DWELL_NS = 10
BB1_PHASE_DEG = 0.0


## 2. Setup


In [ ]:
import math
import sys
from decimal import Decimal
from pathlib import Path

import matplotlib.pyplot as plt

repo_search_start = Path.cwd().resolve()
REPO_ROOT = next(
    candidate
    for candidate in (repo_search_start, *repo_search_start.parents)
    if (candidate / "hardware_capture" / "rf" / "redstone" / "sequence.py").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REDSTONE_DIR = REPO_ROOT / "hardware_capture" / "rf" / "redstone"
OUTPUT_DIR = REDSTONE_DIR / "examples" / "generated" / "pulse_phmod"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPLATE_TPS_PATH = REDSTONE_DIR / "20260630_pulse_phmod_template.tps"
TEMPLATE_TNT_PATH = REDSTONE_DIR / "20260630_pulse_phmod_template.tnt"
OUTPUT_STEM = "20260630_pulse_phmod_bb1_pi_waveform"
OUTPUT_TPS_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.tps"
OUTPUT_TNT_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.tnt"
OUTPUT_JSON_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.tps.json"
OUTPUT_PHASE_TXT_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}_phase_table.txt"

TARGET_PHASE_TABLE = "phmod_pulse"
TARGET_PHASE_LINE = "F1_PhMod"
TARGET_PHASE_EVENTS = (3, 4)

from hardware_capture.rf.redstone import BB1, Sequence

assert TEMPLATE_TPS_PATH.exists(), TEMPLATE_TPS_PATH
assert TEMPLATE_TNT_PATH.exists(), TEMPLATE_TNT_PATH
assert OUTPUT_TPS_PATH.resolve() != TEMPLATE_TPS_PATH.resolve()
assert OUTPUT_TNT_PATH.resolve() != TEMPLATE_TNT_PATH.resolve()


## 3. Load And Inspect The Sequence


In [ ]:
sequence = Sequence.load(TEMPLATE_TPS_PATH)
parameter_values = sequence.parameter_values()
target_references = [
    reference
    for reference in sequence.table_references()
    if reference.table_name == TARGET_PHASE_TABLE
]

target_table = next(table for table in sequence.tables if table.get("name") == TARGET_PHASE_TABLE)

{
    "summary": sequence.summary().to_dict(),
    "target_table_rows_before": len((target_table.get("entries") or "").splitlines()),
    "target_references": target_references,
    "parameters": parameter_values,
}


## 4. Build The BB1 Pi Pulse


In [ ]:
target_pulse_duration_us = Decimal(str(TARGET_PULSE_DURATION_US))
dwell_ns = Decimal(str(DWELL_NS))
if dwell_ns < Decimal("10"):
    raise ValueError("DWELL_NS must be at least 10 ns.")
if dwell_ns % Decimal("10") != 0:
    raise ValueError("DWELL_NS must be on a 10 ns grid.")
if target_pulse_duration_us <= 0:
    raise ValueError("TARGET_PULSE_DURATION_US must be positive.")

points_decimal = target_pulse_duration_us * Decimal("1000") / dwell_ns
if points_decimal != points_decimal.to_integral_exact():
    raise ValueError("TARGET_PULSE_DURATION_US must be an exact integer number of dwell points.")
POINTS = int(points_decimal)
if POINTS % 10:
    raise ValueError("POINTS must be divisible by 10 for the exact split BB1 pi segment ratio.")

DWELL_US = float(dwell_ns / Decimal("1000"))
duration_us_decimal = Decimal(POINTS) * dwell_ns / Decimal("1000")
if duration_us_decimal != target_pulse_duration_us:
    raise ValueError("Computed pulse duration does not exactly match the target pulse duration.")

t_mod_offset_text = str(parameter_values["t_mod_offset"]).strip().lower()
if t_mod_offset_text.endswith("ns"):
    t_mod_offset_us = Decimal(t_mod_offset_text[:-2]) / Decimal("1000")
elif t_mod_offset_text.endswith("n"):
    t_mod_offset_us = Decimal(t_mod_offset_text[:-1]) / Decimal("1000")
elif t_mod_offset_text.endswith("us"):
    t_mod_offset_us = Decimal(t_mod_offset_text[:-2])
elif t_mod_offset_text.endswith("u"):
    t_mod_offset_us = Decimal(t_mod_offset_text[:-1])
else:
    raise ValueError(f"Expected t_mod_offset in ns or us, got {t_mod_offset_text!r}.")

t_pulse_us = target_pulse_duration_us - t_mod_offset_us
if t_pulse_us <= 0:
    raise ValueError("TARGET_PULSE_DURATION_US must be greater than t_mod_offset.")
if t_pulse_us + t_mod_offset_us != target_pulse_duration_us:
    raise ValueError("t_pulse + t_mod_offset does not exactly match the target pulse duration.")

target_duration_text = format(target_pulse_duration_us.normalize(), "f")
t_pulse_text = format(t_pulse_us.normalize(), "f")
duration_us = float(duration_us_decimal)
duration_ms = float(duration_us_decimal / Decimal("1000"))

phase_waveform = BB1(
    angle=180.0,
    phase=BB1_PHASE_DEG,
    split=True,
    repeats=1,
    phase_steps=2**16,
)

segments = phase_waveform.segments()
segment_weights = [1, 2, 4, 2, 1]
segment_point_counts = [POINTS * weight // 10 for weight in segment_weights]
if sum(segment_point_counts) != POINTS:
    raise ValueError("Exact BB1 segment counts do not add up to POINTS.")

freq_offset_text = "0"

{
    "target_pulse_duration_us": target_duration_text,
    "dwell_ns": format(dwell_ns.normalize(), "f"),
    "points": POINTS,
    "duration_us": format(duration_us_decimal.normalize(), "f"),
    "t_mod_offset_us": format(t_mod_offset_us.normalize(), "f"),
    "t_pulse_us": t_pulse_text,
    "timing_check_us": f"{t_pulse_text} + {format(t_mod_offset_us.normalize(), 'f')} = {target_duration_text}",
    "freq_offset": freq_offset_text,
    "segment_point_counts": segment_point_counts,
    "segments": segments,
    "waveform": phase_waveform.summary().to_dict(),
}


## 5. Visualize The BB1 Pulse


In [ ]:
time_ms = [index * DWELL_US * 1.0e-3 for index in range(POINTS)]
base_phase_deg = phase_waveform.phase_values()
if base_phase_deg is None:
    raise ValueError("Phase output is not available.")

phase_deg = []
for value, count in zip(base_phase_deg, segment_point_counts, strict=True):
    phase_deg.extend([value] * count)
if len(phase_deg) != POINTS:
    raise ValueError("Expanded phase output does not match POINTS.")

envelope_i = [math.cos(math.radians(value)) for value in phase_deg]
envelope_q = [math.sin(math.radians(value)) for value in phase_deg]
segment_edges = [0]
for count in segment_point_counts:
    segment_edges.append(segment_edges[-1] + count)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(time_ms, phase_deg, drawstyle="steps-post")
for edge in segment_edges[1:-1]:
    axes[0].axvline(edge * DWELL_US * 1.0e-3, color="0.6", linewidth=0.8, linestyle="--")
axes[0].set_ylabel("Phase (deg)")
axes[0].set_ylim(-5, 365)
axes[0].set_title(f"{TARGET_PHASE_TABLE}: {duration_ms:.3f} ms BB1 pi pulse")
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_ms, envelope_i, drawstyle="steps-post", label="I")
axes[1].plot(time_ms, envelope_q, drawstyle="steps-post", label="Q")
for edge in segment_edges[1:-1]:
    axes[1].axvline(edge * DWELL_US * 1.0e-3, color="0.6", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("Time (ms)")
axes[1].set_ylabel("Unit envelope")
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="upper right")

fig.tight_layout()
plt.show()


## 6. Apply And Save The Sequence


In [ ]:
base_phase_rows = phase_waveform.phase_column(precision=3)
if base_phase_rows is None:
    raise ValueError("Phase output is not available.")

phase_rows = []
for value, count in zip(base_phase_rows, segment_point_counts, strict=True):
    phase_rows.extend([value] * count)
if len(phase_rows) != POINTS:
    raise ValueError("Expanded phase rows do not match POINTS.")

target_table = next(table for table in sequence.tables if table.get("name") == TARGET_PHASE_TABLE)
target_table["entries"] = "\n".join(phase_rows)
target_table["value_type"] = 0
for event_number in TARGET_PHASE_EVENTS:
    sequence.assign_table_to_event(TARGET_PHASE_LINE, event_number, TARGET_PHASE_TABLE)

sequence.set_parameter_value("freq_offset", freq_offset_text)
sequence.set_parameter_value("t_pulse", f"{t_pulse_text}u")

OUTPUT_PHASE_TXT_PATH.write_text("\n".join(phase_rows) + "\n", encoding="utf-8")
sequence.write_json(OUTPUT_JSON_PATH, include_raw_bytes=False)
sequence.write_tps(OUTPUT_TPS_PATH)

reloaded = Sequence.load(OUTPUT_TPS_PATH)
reloaded_table = next(table for table in reloaded.tables if table.get("name") == TARGET_PHASE_TABLE)
assert len((reloaded_table.get("entries") or "").splitlines()) == POINTS
assert reloaded.parameter_values()["freq_offset"] == freq_offset_text
assert reloaded.parameter_values()["t_pulse"] == f"{t_pulse_text}u"

{
    "phase_rows": len(phase_rows),
    "first_10": phase_rows[:10],
    "last_10": phase_rows[-10:],
    "output_tps": OUTPUT_TPS_PATH,
    "output_json": OUTPUT_JSON_PATH,
    "output_phase_table": OUTPUT_PHASE_TXT_PATH,
}


## 7. Save The TNT Copy


In [ ]:
tps_signature = b"PSEQ1.001.18 BIN"
tnt_embedded_signature = b"PSEQ" + (1).to_bytes(4, "little") + b"1.18 BIN"

template_tps_bytes = TEMPLATE_TPS_PATH.read_bytes()
edited_tps_bytes = OUTPUT_TPS_PATH.read_bytes()
template_tnt_bytes = TEMPLATE_TNT_PATH.read_bytes()

if not template_tps_bytes.startswith(tps_signature):
    raise ValueError("Template TPS has an unexpected PSEQ signature.")
if not edited_tps_bytes.startswith(tps_signature):
    raise ValueError("Edited TPS has an unexpected PSEQ signature.")

template_embedded = tnt_embedded_signature + template_tps_bytes[16:]
edited_embedded = tnt_embedded_signature + edited_tps_bytes[16:]
start = template_tnt_bytes.find(template_embedded)
if start < 0:
    raise ValueError("The template TNT does not contain the sibling template TPS payload.")

OUTPUT_TNT_PATH.write_bytes(
    template_tnt_bytes[:start]
    + edited_embedded
    + template_tnt_bytes[start + len(template_embedded) :]
)

restored_embedded = OUTPUT_TNT_PATH.read_bytes()[start : start + len(edited_embedded)]
assert tps_signature + restored_embedded[16:] == edited_tps_bytes

{
    "output_tnt": OUTPUT_TNT_PATH,
    "tnt_bytes": OUTPUT_TNT_PATH.stat().st_size,
    "embedded_tps_offset": start,
}


## 8. Output Summary


In [ ]:
generated_files = [
    OUTPUT_PHASE_TXT_PATH,
    OUTPUT_JSON_PATH,
    OUTPUT_TPS_PATH,
    OUTPUT_TNT_PATH,
]

[
    {"file": path.name, "path": str(path), "bytes": path.stat().st_size}
    for path in generated_files
]
